This script uses LDA to calculate decoding accuracy of neuron populations recorded from different sites. Decoding accuracy is calculated while controlling for number of trials and population size used from each recording. It performs the following steps:

1. Loads relevant data
    - Preprocessed spike times
    - ECoG decoding accuracy
2. Calcualtes decoding accuracy for each recording set under different parameters:
    - Event alignment (peripheral target onset, go cue, movement onset)
    - Number of trials
    - Unit adding
3. Calculates statistics between recordings as a function of ECoG decoding accuracy
4. Plot results and save figures


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import aopy
import os
import pandas as pds
from db import dbfunctions as db
from ipywidgets import interactive, widgets
import scipy
import h5py
from tqdm.auto import tqdm 
import seaborn as sn
import sklearn
from sklearn.decomposition import PCA
from itertools import compress
# import multiprocessing as mp
import time
import math
# from scipy.fft import fft
import glob
from datetime import date

/home/aolab/miniconda3/envs/np_targeting/lib/python3.9/site-packages/one/alf/files.py:10: FutureWarning: `one.alf.files` will be removed in version 3.0. Use `one.alf.path` instead.
  warnings.warn(


In [2]:
aopy.utils.get_memory_available_gb()

654

# Set parameters

In [3]:
from local_functions import load_yaml_params, get_pseudopopulation_raster
subject = 'affi'
param_filename = 'spatiotemporal_structure_params.yaml'
ds_params, data_paths, save_paths, filenames, postproc_params, uq_params, analysis_params, vis_params, target_colors = load_yaml_params(param_filename)
align_events = analysis_params['align_events']

# Load relevant data

## Load ECoG decoding accuracy

In [4]:
ecog_dec_acc = aopy.data.base.load_hdf_group(save_paths['postproc_data'], filenames['ecog_interp'])
day_colors = ecog_dec_acc[subject]['day_colors']

## Load preprocessed neuropixel data

In [5]:
start = time.time()
aopy.utils.release_memory_limit()
df, rasters, preproc_metadata = aopy.data.base.pkl_read(f"{subject}_{filenames['preprocessed']}", save_paths['postproc_data'])
print(f"{np.round((time.time()-start)/60)} min to load preprocessed data")
nrecs = preproc_metadata['nrecs']
recording_site = preproc_metadata['recording_sites'] # will be the same for all align events
implants = ['NPinsert72' if preproc_metadata['implant'][irec] == 'NP_Insert72' else 'NPinsert137' for irec in range(len(preproc_metadata['implant']))] #Rename because name in bmi3d is slightly different (TODO)
dates = np.unique(df['date'])
ntargets = len(np.unique(df['target_idx']))

7.0 min to load preprocessed data


In [6]:
random_units, column_units, depth_units, depth_group_info_by_site, pseudopopulation_metadata, unit_df = aopy.data.base.pkl_read(f"{subject}_{filenames['pseudopopulation']}",  save_paths['postproc_data'])

## Identify stable units

In [7]:
qc_results = aopy.data.base.pkl_read(f"{subject}_{filenames['unit_quality']}", save_paths['postproc_data'])
stable_unit_labels = [qc_results['manual_good_unit_labels'][irec] for irec in range(nrecs)]
stable_unit_idx = [qc_results['manual_good_unit_idx'][irec] for irec in range(nrecs)]
nstable_unit = np.array([len(qc_results['manual_good_unit_idx'][irec]) for irec in range(nrecs)])
neuron_pos = [qc_results['manual_position'][irec] for irec in range(nrecs)]

## Zscore FR

In [8]:
trelevant1 = np.where(preproc_metadata['trial_time_axis']>analysis_params['tstart'])[0][0]
trelevant2 = np.where(preproc_metadata['trial_time_axis']>analysis_params['tend'])[0][0]+1

# Zscore the activity of each neuron (even unstable) across all trials
fr_zscore = {}
for align_event in tqdm(align_events):
    temp_spike_data = [rasters['neural'][align_event][irec][:,np.array(df['good_trial'][df['penetration']==irec]),:][:,:,stable_unit_idx[irec]] for irec in range(nrecs)] # Shape (ntime, ntrial, nunit)
    fr_zscore[align_event] = {'align_spikes_zscore': [], 'align_spikes_zscore_tavg': [], 'align_spikes_zscore_smooth': [], 'target': []}
    [fr_zscore[align_event]['align_spikes_zscore'].append(np.swapaxes((temp_spike_data[irec]-np.mean(temp_spike_data[irec], axis=(0,1)))/np.std(temp_spike_data[irec], axis=(0,1)), 1,2)) for irec in range(nrecs)] #TODO Save rasters flipping dimensions to remove swapaxes
    [fr_zscore[align_event]['align_spikes_zscore_smooth'].append(aopy.postproc.smooth_timeseries_gaus(fr_zscore[align_event]['align_spikes_zscore'][irec], postproc_params['smooth_width'], int(1/postproc_params['bin_width']), postproc_params['smooth_nstd'], conv_mode='valid')) if nstable_unit[irec]>0 else fr_zscore[align_event]['align_spikes_zscore_smooth'].append(np.nan) for irec in range(nrecs)]
    [fr_zscore[align_event]['align_spikes_zscore_tavg'].append(np.mean(fr_zscore[align_event]['align_spikes_zscore'][irec][trelevant1:trelevant2,:,:], axis=0, keepdims=True)) for irec in range(nrecs)]
    [fr_zscore[align_event]['target'].append(np.array(df[df['penetration']==irec]['target_idx'][df['good_trial']])) for irec in range(nrecs)]

  0%|          | 0/1 [00:00<?, ?it/s]

In [9]:
temp_spike_data = [rasters['neural']['RANDOM'][irec][:,:,stable_unit_idx[irec]] for irec in range(nrecs)] # Shape (ntime, ntrial, nunit)    
fr_zscore['RANDOM'] = {'align_spikes_zscore': [], 'align_spikes_zscore_smooth': []}
[fr_zscore['RANDOM']['align_spikes_zscore'].append((temp_spike_data[irec]-np.mean(temp_spike_data[irec], axis=(0,1)))/np.std(temp_spike_data[irec], axis=(0,1))) for irec in range(nrecs)]
[fr_zscore['RANDOM']['align_spikes_zscore_smooth'].append(aopy.postproc.smooth_timeseries_gaus(fr_zscore['RANDOM']['align_spikes_zscore'][irec], postproc_params['smooth_width'], int(1/postproc_params['bin_width']), postproc_params['smooth_nstd'], conv_mode='valid')) if nstable_unit[irec]>0 else fr_zscore['RANDOM']['align_spikes_zscore_smooth'].append(np.nan) for irec in range(nrecs)]

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]

In [10]:
trelevant1 = np.where(preproc_metadata['trial_time_axis']>analysis_params['tstart'])[0][0]
trelevant2 = np.where(preproc_metadata['trial_time_axis']>analysis_params['tend'])[0][0]

# Redefine time axis for smoothed data
kernel_size = np.round((1/postproc_params['bin_width'])*postproc_params['smooth_width']*postproc_params['smooth_nstd']/1000).astype(int)
nt = int((postproc_params['tbefore'] + postproc_params['tafter'])/postproc_params['bin_width'])
smooth_time_axis = preproc_metadata['trial_time_axis'][kernel_size:-kernel_size]
trelevant1_smooth = trelevant1 - kernel_size
trelevant2_smooth = trelevant2 - kernel_size

In [11]:
lda_metadata = {}
lda_metadata['trial_time_axis'] = preproc_metadata['trial_time_axis']
lda_metadata['smooth_time_axis'] = smooth_time_axis
lda_metadata['trelevant1_smooth'] = trelevant1_smooth
lda_metadata['trelevant2_smooth'] = trelevant2_smooth
lda_metadata['trelevant1'] = trelevant1
lda_metadata['trelevant2'] = trelevant2
lda_metadata['recording_site'] = recording_site

# Calculate LDA decoding accuracy

In [12]:
ntime = len(preproc_metadata['trial_time_axis'])
lda_results = {}
lda_results['across_events'] = {}
for align_event in align_events:
    lda_results[align_event] = {}

## All good units in all recordings

In [13]:
# Organize data input to xval_lda
for align_event in tqdm(align_events):
    all_neural_data_list = []
    for irec in range(nrecs):
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)])
        sorted_target_label_mask = np.argsort(temp_target_labels)
        all_neural_data_list.append(fr_zscore[align_event]['align_spikes_zscore_tavg'][irec][:,:,sorted_target_label_mask])
    lda_results[align_event]['all_units_score'], _, lda_results[align_event]['all_units_weight'] = aopy.analysis.windowed_xval_lda_wrapper(np.concatenate(all_neural_data_list, axis=1), temp_target_labels[sorted_target_label_mask], 1/postproc_params['bin_width'], lags=0,  return_weights=True, nfolds=analysis_params['nfolds'])

  0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
# Unit adding spiking
groups_of_nunits_all_recs = np.arange(1,analysis_params['unit_adding_max_units'],analysis_params['unit_adding_unit_step'])
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)]) for irec in range(nrecs)] 
ntime = len(preproc_metadata['trial_time_axis'])

# for align_event in align_events:
for align_event in align_events:
    lda_results[align_event]['unit_adding'] = {}
    lda_results[align_event]['unit_adding']['scores_event'] = []
    for igroup in tqdm(range(len(groups_of_nunits_all_recs))):
        temp_scores = np.zeros((analysis_params['nfolds'], analysis_params['niterations']))*np.nan
        for iiter in range(analysis_params['niterations']):
            pseudopop_raster, target_labels = get_pseudopopulation_raster(unit_df, fr_zscore[align_event]['align_spikes_zscore_tavg'], target_idx_list, nunits=groups_of_nunits_all_recs[igroup])
            a,_ = aopy.analysis.windowed_xval_lda_wrapper(pseudopop_raster, target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
            temp_scores[:,iiter] = a

        lda_results[align_event]['unit_adding']['scores_event'].append(np.mean(temp_scores,axis=0))

  0%|          | 0/60 [00:00<?, ?it/s]

### Single-unit decoding

In [15]:
 # Single channel decoding for spikes
for align_event in align_events:
    lda_results[align_event]['single_ch_decoding'] = []
    lda_results[align_event]['single_ch_modulation_significance'] = []
    for irec in tqdm(range(nrecs)):
        nunits = len(stable_unit_idx[irec])
        temp_single_ch_decoding = np.zeros((nunits,analysis_params['nfolds']))*np.nan
        temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)])
        for ich in range(nunits):
            score, _ = aopy.analysis.windowed_xval_lda_wrapper(fr_zscore[align_event]['align_spikes_zscore_tavg'][irec][:,ich,:][None,:], temp_target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
            temp_single_ch_decoding[ich,:] = score
            
        lda_results[align_event]['single_ch_decoding'].append(temp_single_ch_decoding)

  0%|          | 0/34 [00:00<?, ?it/s]

In [16]:
# Find functionally defined units for all events based on a single channel decoding threshold
for align_event in align_events:
    lda_results[align_event]['single_ch_metadata'] = {}
    single_ch_decoding_event_all = []
    # rec_unit_idx = []
    [single_ch_decoding_event_all.extend(np.mean(lda_results[align_event]['single_ch_decoding'][irec], axis=1)) for irec in range(nrecs)]
    # [rec_unit_idx.extend(np.ones(nstable_unit[irec])*irec) for irec in range(nrecs)]
    lda_results[align_event]['single_ch_decoding_all'] = np.array(single_ch_decoding_event_all)
    lda_results[align_event]['single_ch_metadata']["threshold"] = np.median(single_ch_decoding_event_all) + 1*np.std(single_ch_decoding_event_all)
    print(align_event, lda_results[align_event]['single_ch_metadata']["threshold"])


MOVEMENT ONSET 0.16850519421649618


In [17]:
prep_threshold = lda_results[align_events[0]]['single_ch_metadata']['threshold']
move_threshold = lda_results[align_events[-1]]['single_ch_metadata']['threshold']
lda_results['across_events']['single_ch_metadata'] = {}
lda_results['across_events']['single_ch_metadata']['func_group_mask'] = {}
lda_results['across_events']['single_ch_metadata']['func_group_mask'][analysis_params['functional_group_names'][0]] = np.logical_and(lda_results[align_events[0]]['single_ch_decoding_all'] < prep_threshold, lda_results[align_events[-1]]['single_ch_decoding_all'] < move_threshold)
lda_results['across_events']['single_ch_metadata']['func_group_mask'][analysis_params['functional_group_names'][1]] = np.logical_and(lda_results[align_events[0]]['single_ch_decoding_all'] >= prep_threshold, lda_results[align_events[-1]]['single_ch_decoding_all'] < move_threshold)
lda_results['across_events']['single_ch_metadata']['func_group_mask'][analysis_params['functional_group_names'][2]] = np.logical_and(lda_results[align_events[-1]]['single_ch_decoding_all'] >= move_threshold, lda_results[align_events[0]]['single_ch_decoding_all'] < prep_threshold)
lda_results['across_events']['single_ch_metadata']['func_group_mask'][analysis_params['functional_group_names'][3]] = np.logical_and(lda_results[align_events[0]]['single_ch_decoding_all'] >= prep_threshold, lda_results[align_events[-1]]['single_ch_decoding_all'] >= move_threshold)


In [18]:
# Get rasters of high decoding units for all events
nunits_per_group = np.min([np.sum(lda_results['across_events']['single_ch_metadata']['func_group_mask'][label]) for label in analysis_params['functional_group_names']])
single_ch_groups = {}

for align_event in align_events:  
    single_ch_decoding_prep_data = []
    
    for irec in tqdm(range(nrecs)):
        if len(stable_unit_idx[irec]) > 0:
            # Organize trials to be consistent across recordings
            ordered_target_labels_idx = np.argsort(np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)]))
            single_ch_pp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)])[ordered_target_labels_idx]
            single_ch_decoding_prep_data.append(fr_zscore[align_event]['align_spikes_zscore_tavg'][irec][:,:,ordered_target_labels_idx])

    single_ch_decoding_prep_data = np.concatenate(single_ch_decoding_prep_data, axis=1)

    single_ch_groups[align_event] = single_ch_decoding_prep_data

  0%|          | 0/34 [00:00<?, ?it/s]

In [19]:
if len(align_events) > 1: # Comparing different task events requires multiple task events
    lda_results['across_events']['single_ch'] = {}    
    for idec_group, decoding_group in enumerate(tqdm(analysis_params['functional_group_names'])):
        temp_scores = np.zeros((len(align_events), len(align_events), analysis_params['niterations']))*np.nan
        for ievent_train, align_event_train in enumerate(align_events):
            for ievent_test, align_event_test in enumerate(align_events):
                for iiter in range(analysis_params['niterations']):
                    unit_list = np.where(lda_results['across_events']['single_ch_metadata']['func_group_mask'][decoding_group])[0]
                    random_unit_idx = np.random.choice(unit_list, analysis_params['nunits_across_events'], replace=False)

                    input_data_train = single_ch_groups[align_event_train][0,random_unit_idx,:].T
                    input_data_test = single_ch_groups[align_event_test][0,random_unit_idx,:].T

                    # Train LDA model on data from train event
                    lda = sklearn.discriminant_analysis.LinearDiscriminantAnalysis(solver='eigen', shrinkage='auto')
                    lda.fit(input_data_train, single_ch_pp_target_labels)
                    temp_scores[ievent_train, ievent_test, iiter] = lda.score(input_data_test, single_ch_pp_target_labels)

        # lda_results['across_events']['pseudopopulations']['column_scores'].append(temp_scores)
        lda_results['across_events']['single_ch'][decoding_group] = temp_scores

## All good units in each recording

In [20]:
for align_event in align_events:
    lda_results[align_event]['all_unit_scores'] = []
    for irec in tqdm(range(nrecs)):
        if nstable_unit[irec] > 0:
            units_to_use = stable_unit_idx[irec]
            temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)])
            score, _ = aopy.analysis.windowed_xval_lda_wrapper(fr_zscore[align_event]['align_spikes_zscore_tavg'][irec][:,:,:], temp_target_labels,  1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])        
            lda_results[align_event]['all_unit_scores'].append(score)
        else:
            lda_results[align_event]['all_unit_scores'].append([])

  0%|          | 0/34 [00:00<?, ?it/s]

## Neuron number matched

In [21]:
for align_event in tqdm(align_events):
    lda_results[align_event]['penetration_score'] = []
    for irec in tqdm(range(nrecs)):
        if len(stable_unit_idx[irec]) >= analysis_params['nunits_column']:
            temp_target_labels = np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)])
            temp_lda_neural = np.zeros((analysis_params['niterations'], analysis_params['nfolds']))*np.nan
            nunits = len(stable_unit_idx[irec])
            for ii in range(analysis_params['niterations']):
                unit_idx_temp = np.arange(nunits) if nunits == analysis_params['nunits_column'] else np.random.choice(np.arange(nunits), size=analysis_params['nunits_column'], replace=False)                
                score, _ = aopy.analysis.windowed_xval_lda_wrapper(fr_zscore[align_event]['align_spikes_zscore_tavg'][irec][:,unit_idx_temp,:], temp_target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
                temp_lda_neural[ii,:] = score

            lda_results[align_event]['penetration_score'].append(temp_lda_neural)
        else:
            print(f"Not enough units in recording {irec}, site {recording_site[irec]}")
            lda_results[align_event]['penetration_score'].append([])

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/34 [00:00<?, ?it/s]

Not enough units in recording 3, site 58
Not enough units in recording 10, site 70
Not enough units in recording 13, site 31
Not enough units in recording 15, site 21
Not enough units in recording 16, site 69
Not enough units in recording 20, site 31
Not enough units in recording 29, site 58
Not enough units in recording 30, site 17
Not enough units in recording 31, site 58
Not enough units in recording 32, site 18
Not enough units in recording 33, site 49


## Pseudopopulations

In [22]:
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)]) for irec in range(nrecs)] 
for align_event in align_events:
    lda_results[align_event]['pseudopopulations'] = {}

### Random

In [23]:
# Use only a single time window to compute decoding accuracy
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['random_scores'] = []
    for igroup in tqdm(range(len(random_units))):
        pseudopop_raster, target_labels = get_pseudopopulation_raster(random_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore_tavg'], target_idx_list, nunits=analysis_params['nunits_column'])
        temp_scores, _ = aopy.analysis.windowed_xval_lda_wrapper(pseudopop_raster, target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
        lda_results[align_event]['pseudopopulations']['random_scores'].append(temp_scores)

  0%|          | 0/1000 [00:00<?, ?it/s]

### Column

In [24]:
# Use only a single time window to compute decoding accuracy
for align_event in tqdm(align_events):
    lda_results[align_event]['pseudopopulations']['column_scores'] = []
    for igroup in tqdm(range(len(column_units))):
        if len(column_units[igroup]) >= analysis_params['nunits_column']:
            temp_scores = np.zeros((analysis_params['niterations'], analysis_params['nfolds']))*np.nan
            for iiter in range(analysis_params['niterations']):
                pseudopop_raster, target_labels = get_pseudopopulation_raster(column_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore_tavg'], target_idx_list, nunits=analysis_params['nunits_column'])
                score, _ = aopy.analysis.windowed_xval_lda_wrapper(pseudopop_raster, target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
                temp_scores[iiter,:] = score

            lda_results[align_event]['pseudopopulations']['column_scores'].append(np.mean(temp_scores,axis=0))

        else:
            temp_site = column_units[igroup]['rec_site'][0]
            print(f'Exception: Not enough units in site:{temp_site}')
            lda_results[align_event]['pseudopopulations']['column_scores'].append([])

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/26 [00:00<?, ?it/s]

Exception: Not enough units in site:21
Exception: Not enough units in site:18
Exception: Not enough units in site:69
Exception: Not enough units in site:58
Exception: Not enough units in site:48
Exception: Not enough units in site:58


In [34]:
len(fr_zscore[align_event]['align_spikes_zscore_tavg']), fr_zscore[align_event]['align_spikes_zscore_tavg'][0].shape

(34, (1, 82, 240))

In [25]:
# Unit adding for each column
# groups_of_nunits = np.arange(1, 30, 1)
# # for align_event in align_events:
# align_event = align_events[-1]
# lda_results[align_event]['unit_adding']['scores_recording'] = {}
# # lda_results[align_event]['pseudopopulations']['column_weights'] = []
# for igroup in tqdm(range(len(column_units))):
#     lda_results[align_event]['unit_adding']['scores_recording'][igroup] = []
#     for group_size in groups_of_nunits:
#         if group_size <= len(column_units[igroup].reset_index()):
#             temp_scores = np.zeros((nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             temp_weights = np.zeros((ntargets, group_size,  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             for iiter in range(pseudopopulation_metadata['nrandom_groups']):
#                 pseudopop_raster, target_labels = get_pseudopopulation_raster(column_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=group_size)
#                 input_data = np.swapaxes(np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2]), 1,2)
#                 a, b = xval_lda(input_data, target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=False, return_weights=True, nfolds=nfolds)
#                 temp_scores[:,iiter] = a
#                 # temp_weights[:,:,:,iiter] = b

#             lda_results[align_event]['unit_adding']['scores_recording'][igroup].append(temp_scores)
#         else:
#             # print(f"Not enough units at recroding site {column_units[igroup]['rec_site'][0]} - Group size: {group_size}")
#             continue

#         # lda_results[align_event]['pseudopopulations']['column_weights'].append(np.mean(temp_weights,axis=3))

In [26]:
# # Compile into arrays before plotting
# prep_to_mov_decoding = []
# mov_to_prep_decoding = []
# prep_to_mov_decoding_sd = []
# mov_to_prep_decoding_sd = []
# for irec in range(len(column_units)):
#     try:
#         prep_to_mov_decoding.append(np.mean(lda_results['across_events']['pseudopopulations']['column_scores'][irec], axis=(2))[0,2])
#         mov_to_prep_decoding.append(np.mean(lda_results['across_events']['pseudopopulations']['column_scores'][irec], axis=(2))[2,0])
#         prep_to_mov_decoding_sd.append(np.std(lda_results['across_events']['pseudopopulations']['column_scores'][irec], axis=(2))[0,2])
#         mov_to_prep_decoding_sd.append(np.std(lda_results['across_events']['pseudopopulations']['column_scores'][irec], axis=(2))[2,0])
#     except:
#         prep_to_mov_decoding.append([])
#         mov_to_prep_decoding.append([])
#         prep_to_mov_decoding_sd.append([])
#         mov_to_prep_decoding_sd.append([])

In [27]:
# # By column split between superficial and deep units
# print(pseudopopulation_metadata['nrandom_groups'])
# superficial_depth_range = (-300, 1200)
# deep_depth_range = (1200, 4000)
# for align_event in tqdm(align_events):
#     nunits_2_use = 5
#     align_event = align_events[-1]
#     lda_results[align_event]['pseudopopulations']['column_scores_superficial'] = []
#     lda_results[align_event]['pseudopopulations']['column_weights_superficial'] = []
#     lda_results[align_event]['pseudopopulations']['column_scores_deep'] = []
#     lda_results[align_event]['pseudopopulations']['column_weights_deep'] = []
#     for igroup in tqdm(range(len(column_units))):
#         superficial_unit_df = column_units[igroup].loc[(column_units[igroup]['rel_depth'] > superficial_depth_range[0]) & (column_units[igroup]['rel_depth'] < superficial_depth_range[1]),:].reset_index()
#         deep_unit_df = column_units[igroup].loc[(column_units[igroup]['rel_depth'] > deep_depth_range[0]) & (column_units[igroup]['rel_depth'] < deep_depth_range[1]),:].reset_index()

#         if len(superficial_unit_df) >= nunits_2_use:
#             temp_scores = np.zeros((nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             temp_weights = np.zeros((ntargets,  nunits_2_use,  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             for iiter in range(pseudopopulation_metadata['nrandom_groups']):
#                 pseudopop_raster, target_labels = get_pseudopopulation_raster(superficial_unit_df, fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=nunits_2_use)
#                 input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
#                 a, b = xval_lda(np.swapaxes(input_data, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=False, return_weights=True, nfolds=nfolds)
#                 temp_scores[:,iiter] = a
#                 temp_weights[:,:,:,iiter] = b

#             lda_results[align_event]['pseudopopulations']['column_scores_superficial'].append(np.mean(temp_scores,axis=0))
#             lda_results[align_event]['pseudopopulations']['column_weights_superficial'].append(np.mean(temp_weights,axis=2))

#         else:
#             print(f'Not enough superficial units. Group {igroup}')
#             lda_results[align_event]['pseudopopulations']['column_scores_superficial'].append([])
#             lda_results[align_event]['pseudopopulations']['column_weights_superficial'].append([])


#         if len(deep_unit_df) >= nunits_2_use:
#             temp_scores = np.zeros((nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             temp_weights = np.zeros((ntargets,  nunits_2_use,  nfolds, pseudopopulation_metadata['nrandom_groups']))*np.nan
#             for iiter in range(pseudopopulation_metadata['nrandom_groups']):
#                 pseudopop_raster, target_labels = get_pseudopopulation_raster(deep_unit_df, fr_zscore[align_event]['align_spikes_zscore'], stable_unit_idx, target_idx_list, nunits=nunits_2_use)
#                 input_data = np.mean(pseudopop_raster[trelevant1:trelevant2,:,:], axis=0).reshape(1,pseudopop_raster.shape[1], pseudopop_raster.shape[2])
#                 a, b = xval_lda(np.swapaxes(input_data, 1,2), target_labels, 1/preproc_metadata['spike_bin_width'], lags=0, smooth_timeseries=False, return_weights=True, nfolds=nfolds)
#                 temp_scores[:,iiter] = a
#                 temp_weights[:,:,:,iiter] = b

#             lda_results[align_event]['pseudopopulations']['column_scores_deep'].append(np.mean(temp_scores,axis=0))
#             lda_results[align_event]['pseudopopulations']['column_weights_deep'].append(np.mean(temp_weights,axis=2))

#         else:
#             print(f'Not enough deep units. Group {igroup}')
#             lda_results[align_event]['pseudopopulations']['column_scores_deep'].append([])
#             lda_results[align_event]['pseudopopulations']['column_weights_deep'].append([])


### Depth

In [28]:
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['depth_scores'] = []
    for igroup in tqdm(range(len(depth_units))):
        temp_scores = np.zeros((analysis_params['niterations'], analysis_params['nfolds']))*np.nan
        for iiter in range(analysis_params['niterations']):
            pseudopop_raster, target_labels = get_pseudopopulation_raster(depth_units[igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore_tavg'], target_idx_list, nunits=analysis_params['nunits_lda_depth'])
            score, _ = aopy.analysis.windowed_xval_lda_wrapper(pseudopop_raster, target_labels,  1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
            temp_scores[iiter, :] = score
        lda_results[align_event]['pseudopopulations']['depth_scores'].append(temp_scores)


  0%|          | 0/12 [00:00<?, ?it/s]

In [31]:
niterations = analysis_params['niterations']
ntime = len(preproc_metadata['trial_time_axis'])
target_idx_list = [np.array(df['target_idx'][df['good_trial']*(df['penetration']==irec)]) for irec in range(nrecs)] 
for align_event in align_events:
    lda_results[align_event]['pseudopopulations']['depth_scores_by_site'] = {}
    for isite, site in enumerate(tqdm(list(depth_group_info_by_site.keys()))):
        lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site] = []
        for igroup in range(len(depth_group_info_by_site[site])):
            temp_scores = np.zeros((analysis_params['nfolds'], niterations))*np.nan
            if len(depth_group_info_by_site[site][igroup]) >= analysis_params['nunits_lda_depth_site']:
                for iiter in range(niterations):
                    pseudopop_raster, target_labels = get_pseudopopulation_raster(depth_group_info_by_site[site][igroup].reset_index(), fr_zscore[align_event]['align_spikes_zscore_tavg'], target_idx_list, nunits=analysis_params['nunits_lda_depth_site'])
                    scores, _ = aopy.analysis.base.windowed_xval_lda_wrapper(pseudopop_raster, target_labels, 1/postproc_params['bin_width'], lags=0, return_weights=False, nfolds=analysis_params['nfolds'])
                    temp_scores[:,iiter] = scores

            lda_results[align_event]['pseudopopulations']['depth_scores_by_site'][site].append(np.mean(temp_scores,axis=0))

  0%|          | 0/26 [00:00<?, ?it/s]

# Save analysis results

In [32]:
start = time.time()
aopy.data.base.pkl_write(f"{subject}_{filenames['lda_result']}", (lda_results, fr_zscore, lda_metadata), save_paths['postproc_data'])
print(f"It took {np.round(time.time() - start, 3)}s to save")

It took 15.015s to save
